# Day 05 — Exception Handling + Retry + logging
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `try/except` · catching specific exceptions · `finally` · `raise` · built-in exception types · the `logging` module · log levels · retry with exponential backoff

> **Connection to Day 02:** The `while` retry loop from Day 02 is now production-grade — wrapped in `try/except` with logging at every step.

> **Note on OOP:** Custom exception *classes* (using `class MyError(Exception)`) require OOP knowledge covered in Day 08. Today we use Python's **built-in exception types** only — `ValueError`, `KeyError`, `json.JSONDecodeError`, etc.


---


## 1 — Imports


In [ ]:
import json       # json.JSONDecodeError — raised when JSON is invalid
import logging    # structured logging — replaces print() in production
import time       # time.sleep() — pause between retries


---


## 2 — What is an Exception?

An **exception** is an error that occurs at runtime — when something unexpected happens that Python cannot handle by itself.

Without handling exceptions, one bad CSV row or one invalid LLM response crashes your entire service. With `try/except`, you handle the error gracefully and keep running.

### Common built-in exceptions you will see every day

| Exception | When it's raised | LLM engineering example |
|-----------|-----------------|------------------------|
| `KeyError` | Dict key doesn't exist | `row['customer_id']` when column is missing |
| `ValueError` | Value has wrong format | `int('abc')` — can't convert |
| `TypeError` | Wrong type passed | `len(None)` — None has no length |
| `FileNotFoundError` | File path doesn't exist | `open('prompts/missing.txt')` |
| `json.JSONDecodeError` | String is not valid JSON | `json.loads('not json')` |
| `ZeroDivisionError` | Division by zero | `total / count` when count is 0 |
| `IndexError` | List index out of range | `items[10]` on a 3-item list |
| `AttributeError` | Object has no such attribute | `None.strip()` |


---


## 3 — try / except — Basic Syntax

```python
try:
    # the risky code — might raise an exception
    result = int(user_input)
except ValueError:
    # runs ONLY if the try block raised a ValueError
    print('Not a number')
```

**Python checks the `except` clauses top to bottom — most specific first.**  
If no exception is raised, the `except` block is skipped entirely.


In [ ]:
# ── 1. Catching a single exception type ─────────────────────
def safe_to_int(value):
    """Convert value to int. Returns None if conversion fails."""
    try:
        return int(value)   # may raise ValueError or TypeError
    except ValueError:
        # ValueError: value has wrong format → int('abc') → ValueError
        print(f"  ValueError: '{value}' cannot be converted to int")
        return None

print(safe_to_int('1001'))  # 1001  — no exception
print(safe_to_int('abc'))   # None  — ValueError caught
print(safe_to_int('3.14'))  # None  — ValueError caught (float str → int fails)


In [ ]:
# ── 2. Catching multiple exception types ─────────────────────
def get_customer_name(customer: dict, key: str):
    """Get a field from customer dict safely."""
    try:
        return customer[key].strip().title()   # KeyError if key missing, AttributeError if None
    except KeyError:
        # KeyError: key doesn't exist in the dict
        print(f"  KeyError: key '{key}' not found in customer dict")
        return 'Unknown'
    except AttributeError:
        # AttributeError: value exists but is None — None.strip() fails
        print(f"  AttributeError: value for '{key}' is None")
        return 'Unknown'

customer = {'customer_id': '1001', 'name': 'prudhvi akella', 'email': None}

print(get_customer_name(customer, 'name'))      # 'Prudhvi Akella'
print(get_customer_name(customer, 'email'))     # AttributeError — None.strip()
print(get_customer_name(customer, 'city'))      # KeyError — key missing


In [ ]:
# ── 3. Catching multiple types in one except line ────────────
def parse_csv_field(value: str, target_type: type):
    """Convert a CSV string field to target_type."""
    try:
        return target_type(value)
    except (ValueError, TypeError):
        # (ValueError, TypeError) in a tuple — catches both in one clause
        print(f"  Could not convert '{value}' to {target_type.__name__}")
        return None

print(parse_csv_field('205.21', float))   # 205.21
print(parse_csv_field('abc',    float))   # None  — ValueError
print(parse_csv_field(None,     float))   # None  — TypeError


---


## 4 — Reading the Exception Message with `as`

Add `as e` to capture the exception object. This gives you the error message and lets you log it.

```python
except ValueError as e:
    print(f'Error: {e}')   # e contains the error message string
```


In [ ]:
# except ExceptionType as e — e holds the exception object
# str(e) or f'{e}' gives the error message

bad_json = '{"category": "TRACK_ORDER", missing_quote: "high"}'  # invalid JSON

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as e:
    # json.JSONDecodeError: raised when json.loads() gets invalid JSON
    # e.msg        → short error message
    # e.lineno     → line number where error occurred
    # e.colno      → column number where error occurred
    print(f"JSONDecodeError: {e}")
    print(f"  message : {e.msg}")
    print(f"  position: line {e.lineno}, column {e.colno}")
    parsed = {}   # safe fallback — empty dict

print(f"parsed = {parsed}")


---


## 5 — `finally` — Always Runs

`finally` runs **no matter what** — whether the `try` block succeeded, raised an exception, or was caught by `except`.  
Use it for **cleanup**: closing files, recording timing, releasing locks.

```python
try:
    result = risky_operation()
except SomeError as e:
    handle_error(e)
finally:
    cleanup()   # ALWAYS runs — success or failure
```


In [ ]:
import time

def parse_with_timing(raw_response: str) -> dict:
    """Parse LLM JSON and always log how long it took."""
    start = time.perf_counter()   # start timer before the risky code
    result = {}

    try:
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
        result = json.loads(cleaned)
        print(f"  Parse succeeded: {result}")

    except json.JSONDecodeError as e:
        print(f"  Parse FAILED: {e.msg}")
        result = {}   # safe empty fallback

    finally:
        # finally ALWAYS runs — we always want to record the timing
        elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
        print(f"  Elapsed: {elapsed_ms}ms  (finally block — always runs)")

    return result

# Test 1: valid JSON
print("Valid JSON:")
parse_with_timing('{"category": "TRACK_ORDER", "confidence": "high"}')
print()

# Test 2: invalid JSON
print("Invalid JSON:")
parse_with_timing('{bad json here}')


---


## 6 — `raise` — Re-raising and Raising Exceptions

| Pattern | When to use |
|---------|------------|
| `raise` | Re-raise the current exception (keeps original traceback) |
| `raise ValueError('message')` | Raise a new exception with a custom message |
| `raise ValueError('msg') from e` | Raise a new exception, chain it to the original |

Use `raise` when the current function cannot handle the error itself and the **caller** needs to know about it.


In [ ]:
def validate_customer_id(customer_id: str) -> int:
    """Convert customer_id string to int, raise ValueError with a clear message if invalid."""
    try:
        cid = int(customer_id)
    except ValueError as e:
        # Don't silently return None — raise a clear error so the caller knows
        raise ValueError(f"customer_id must be a number, got: '{customer_id}'") from e
        # 'from e' chains the original exception — Python shows both in the traceback

    if cid <= 0:
        # Raise a ValueError ourselves when the value is logically invalid
        raise ValueError(f"customer_id must be positive, got: {cid}")

    return cid

# Test: valid id
print(validate_customer_id('1001'))   # 1001

# Test: invalid id
try:
    validate_customer_id('abc')
except ValueError as e:
    print(f"Caught: {e}")

# Test: negative id
try:
    validate_customer_id('-5')
except ValueError as e:
    print(f"Caught: {e}")


---


## 7 — The `logging` Module

### Why not just use `print()`?

| | `print()` | `logging` |
|-|-----------|----------|
| Shows timestamp | No | Yes |
| Shows severity level | No | Yes (INFO, WARNING, ERROR) |
| Can be turned off | No — always prints | Yes — set level to filter |
| Goes to file + console | No | Yes — multiple handlers |
| Production-ready | No | Yes |

### Log levels — in order of severity

| Level | Number | When to use |
|-------|--------|------------|
| `DEBUG` | 10 | Detailed diagnostics — only during development |
| `INFO` | 20 | Normal operation — function started/completed |
| `WARNING` | 30 | Unexpected but handled — missing field, slow response |
| `ERROR` | 40 | Something failed — exception caught |
| `CRITICAL` | 50 | Serious failure — service about to stop |

> Setting level to `WARNING` means only WARNING, ERROR, and CRITICAL messages appear — DEBUG and INFO are filtered out.


In [ ]:
import logging

# basicConfig sets up the root logger — call ONCE at the top of your application
# format = what each log line looks like
# level  = minimum severity to display (INFO means DEBUG is hidden)
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)-8s %(message)s',
    force=True,   # force=True lets us reconfigure in Jupyter (normally call once)
)

log = logging.getLogger(__name__)   # get a logger named after this module

# The five log levels
log.debug(   'DEBUG   — detailed info, only shown in development')
log.info(    'INFO    — normal operation: function started, request received')
log.warning( 'WARNING — unexpected but handled: missing CSV column')
log.error(   'ERROR   — something failed: json.JSONDecodeError caught')
log.critical('CRITICAL— serious failure: database unreachable')


In [ ]:
# ── Logging in practice — with exception handling ────────────
def safe_parse_llm_json(raw_response: str) -> dict:
    """
    Parse LLM JSON response with logging at each step.
    No OOP — uses only built-in json.JSONDecodeError.
    """
    log.info(f"Parsing LLM response ({len(raw_response)} chars)")
    start = time.perf_counter()

    try:
        # Step 1: strip markdown fences if present
        cleaned = raw_response.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
            log.debug('Stripped markdown fences from response')

        # Step 2: parse JSON string → Python dict
        # json.loads() raises json.JSONDecodeError if string is not valid JSON
        parsed = json.loads(cleaned)
        log.debug(f"Parsed successfully. Keys: {list(parsed.keys())}")
        return parsed

    except json.JSONDecodeError as e:
        # json.JSONDecodeError is a built-in exception — no OOP needed
        log.error(f"JSON parse failed at line {e.lineno} col {e.colno}: {e.msg}")
        log.error(f"Raw response was: {raw_response[:100]}")
        return {}   # return empty dict — caller checks with .get()

    finally:
        elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
        log.info(f"Parse finished in {elapsed_ms}ms")

# Demo
print("\n--- Test 1: valid JSON ---")
result = safe_parse_llm_json('{"category": "TRACK_ORDER", "confidence": "high"}')
print(f"Result: {result}")

print("\n--- Test 2: fenced JSON ---")
result2 = safe_parse_llm_json('```json\n{"intent": "CANCEL"}\n```')
print(f"Result: {result2}")

print("\n--- Test 3: invalid JSON ---")
result3 = safe_parse_llm_json('{bad json here}')
print(f"Result: {result3}")


---


## 8 — Applying try/except to Real Data: CSV Row Parsing

CSV data from `DictReader` always gives strings — and columns can be missing or malformed.  
Wrap conversion in `try/except` so one bad row doesn't crash processing of thousands of good rows.


In [ ]:
def parse_customer_csv_row(row: dict) -> dict:
    """
    Parse and validate a single row from customers.csv.
    Returns a typed dict on success, empty dict on failure.
    Logs a warning for any row that fails — never crashes.
    """
    try:
        return {
            'customer_id': int(row['customer_id']),          # str → int
            'name'       : f"{row['first_name']} {row['last_name']}".strip(),
            'email'      : row['email'].lower().strip(),
            'city'       : row['city'],
        }

    except KeyError as e:
        # KeyError: a required column is missing from this CSV row
        # Usually means the CSV schema changed — log it and skip the row
        log.warning(f"Missing column in customer row: {e}. Row: {row}")
        return {}

    except ValueError as e:
        # ValueError: a column value can't be converted to the expected type
        # e.g. int('N/A') → ValueError
        log.warning(f"Type conversion failed for customer row: {e}. Row: {row}")
        return {}


# Demo
print("--- Valid row ---")
valid_row = {'customer_id': '1001', 'first_name': 'Prudhvi', 'last_name': 'Akella',
             'email': '  PRUDHVI@Example.COM  ', 'city': 'Rajahmundry'}
print(parse_customer_csv_row(valid_row))

print("\n--- Missing column (KeyError) ---")
missing_col = {'customer_id': '1002', 'first_name': 'Ravi'}   # last_name, email, city missing
print(parse_customer_csv_row(missing_col))

print("\n--- Bad type (ValueError) ---")
bad_type = {'customer_id': 'N/A', 'first_name': 'Ananya', 'last_name': 'Sharma',
            'email': 'a@b.com', 'city': 'Bangalore'}
print(parse_customer_csv_row(bad_type))


---


## 9 — Retry with Exponential Backoff + Logging

This is the Day 02 `while` retry loop now made production-grade:  
- `try/except` inside the `while` loop catches specific errors  
- `logging` records every attempt, wait, and outcome  
- Some errors (auth failure) should **not** be retried — `raise` immediately  
- Some errors (rate limit, timeout) should be retried after a wait  

```
attempt=0 → API call fails (rate limit) → wait 0.1s → attempt=1
attempt=1 → API call fails (timeout)    → wait 0.2s → attempt=2
attempt=2 → API call succeeds           → return response
```


In [ ]:
def call_llm_api_with_retry(
    prompt: str,
    model: str = 'gpt-4o',
    max_retries: int = 3,
) -> str:
    """
    Call an LLM API with retry logic and logging.
    Different errors get different treatment:
      - ConnectionError   → retry after wait (recoverable)
      - TimeoutError      → retry after wait (recoverable)
      - PermissionError   → raise immediately (auth failure — retrying won't help)
      - json.JSONDecodeError → retry with cleaner prompt instruction
    """
    attempt    = 0
    last_error = None

    # ── ITERATION TRACE (3 retries, succeeds on attempt 3):
    # attempt=0 → _simulate_llm_call raises ConnectionError
    #             wait = 0.1 * (2**0) = 0.1s
    #             attempt = 1
    #
    # attempt=1 → _simulate_llm_call raises TimeoutError
    #             wait = 0.1 * (2**1) = 0.2s
    #             attempt = 2
    #
    # attempt=2 → _simulate_llm_call succeeds
    #             returns '{"category": "TRACK_ORDER", ...}'
    while attempt < max_retries:
        try:
            log.info(f"LLM call attempt {attempt + 1}/{max_retries} | model={model}")
            response = _simulate_llm_call(prompt, attempt)
            log.info(f"Succeeded on attempt {attempt + 1}")
            return response   # success — exit loop immediately

        except PermissionError as e:
            # Auth errors are NOT retryable — fail immediately
            # 'raise' re-raises the current exception with its original traceback
            log.error(f"Auth failed — check your API key: {e}")
            raise

        except (ConnectionError, TimeoutError) as e:
            # Transient errors — wait and retry with exponential backoff
            wait = 0.1 * (2 ** attempt)   # 0.1s, 0.2s, 0.4s, 0.8s
            log.warning(f"Attempt {attempt + 1} failed ({type(e).__name__}). Waiting {wait:.1f}s")
            time.sleep(wait)
            last_error = e

        except json.JSONDecodeError as e:
            # Bad JSON — add stricter instruction and retry
            log.warning(f"LLM returned invalid JSON. Retrying with stricter instruction.")
            prompt     = prompt + '\n\nIMPORTANT: Respond ONLY with valid JSON.'
            last_error = e

        attempt += 1   # MUST increment — otherwise infinite loop

    # All retries exhausted
    log.error(f"All {max_retries} attempts failed. Last error: {last_error}")
    raise RuntimeError(f"LLM call failed after {max_retries} attempts") from last_error


In [ ]:
def _simulate_llm_call(prompt: str, attempt: int) -> str:
    """
    Simulates a real LLM API call for demo purposes.
    attempt 0 → ConnectionError (rate limit)
    attempt 1 → TimeoutError
    attempt 2 → success
    """
    if attempt == 0:
        raise ConnectionError('HTTP 429: Too Many Requests')
    elif attempt == 1:
        raise TimeoutError('Request timed out after 30s')
    else:
        return '{"category": "TRACK_ORDER", "confidence": "high"}'


# Demo
print("--- Retry demo ---")
try:
    response = call_llm_api_with_retry('Where is order #3042?', max_retries=3)
    print(f"Final response: {response}")
except RuntimeError as e:
    print(f"All retries exhausted: {e}")


---


## 10 — Filtering Log Levels

In production you set the level to `WARNING` so only problems appear — not every INFO or DEBUG message.  
In development you use `DEBUG` to see everything.


In [ ]:
# Set level to WARNING — only WARNING, ERROR, CRITICAL appear
logging.basicConfig(
    level=logging.WARNING,
    format='%(levelname)-8s %(message)s',
    force=True,
)
log = logging.getLogger(__name__)

log.debug(  'DEBUG   — NOT shown (below WARNING level)')
log.info(   'INFO    — NOT shown (below WARNING level)')
log.warning('WARNING — shown')
log.error(  'ERROR   — shown')

print()
# Reset to DEBUG for the rest of the notebook
logging.basicConfig(level=logging.DEBUG, format='%(levelname)-8s %(message)s', force=True)
log = logging.getLogger(__name__)


---


## Summary — Day 05

| Concept | Key rule / syntax |
|---------|-------------------|
| `try/except` | Wrap risky code. `except` runs only if that error type is raised. |
| Catch specific first | `except ValueError` before `except Exception` — most specific first |
| `except (A, B)` | Catch multiple types in one clause |
| `except Error as e` | `e` holds the exception object and message |
| `finally` | ALWAYS runs — use for cleanup, timing, closing resources |
| `raise` | Re-raise current exception (keeps traceback) |
| `raise ValueError('msg')` | Raise a new exception with a clear message |
| `raise X from e` | Chain new exception to original |
| `KeyError` | Dict key missing — `row['col']` when column absent |
| `ValueError` | Wrong format — `int('abc')` |
| `json.JSONDecodeError` | Invalid JSON — `json.loads('bad')` |
| `FileNotFoundError` | File path doesn't exist |
| `logging.basicConfig()` | Call ONCE at startup. Sets format and minimum level. |
| Log levels | DEBUG < INFO < WARNING < ERROR < CRITICAL |
| `log.info()` / `log.warning()` / `log.error()` | Use the right level for the right event |
| Retry pattern | `while attempt < max_retries` + `try/except` + `time.sleep(0.1 * 2**attempt)` |
| Don't retry auth errors | `raise` immediately on `PermissionError` — retrying won't help |
| OOP custom exceptions | Covered in Day 08 after OOP is introduced |

**Next:** Day 06 — Decorators and Higher-Order Functions  
`@decorator` syntax · `functools.wraps` · timing decorators · retry as a decorator
